In [1]:
import os
import sys
sys.path.append('..')
# sys.path.append('../models/DarkStreamTVT')
import re
import glob
import tqdm
import torch
import librosa
import pandas as pd
from src.modules import SpeakerModule

In [2]:
speaker_encoder = SpeakerModule(device="cuda" if torch.cuda.is_available() else "cpu")

/data/waris/installations/darkstream/lib/python3.10/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
/data/waris/installations/darkstream/lib/python3.10/site-packages/speechbrain/utils/checkpoints.py:200: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they ar

In [3]:
model_name = "DarkStream-quant-TVT-nogate-ns" # Options: DarkStream-bottleneck-TVT, DarkStream-logits-TVT, DarkStream-quant-TVT

In [4]:
data_dir = {
    # source datasets
    'artic': "/data/mtseng/L2Artic/all_data_for_ac_vc",
    'l2artic': "/data/mtseng/L2Artic/all_data_for_ac_vc",
    'vctk': "/data/mtseng/VCTK-Corpus/wav48",
    
    # target datasets
    'emime-eng': "/data/mtseng/EMIME/UEDIN_bilingual_96kHz_data/English_talkers",
    'emime-fin': "/data/mtseng/EMIME/UEDIN_bilingual_96kHz_data/Finnish_talkers",
    'emime-ger': "/data/mtseng/EMIME/UEDIN_bilingual_96kHz_data/German_talkers",
    'emime-man': "/data/mtseng/EMIME/UEDIN_bilingual_96kHz_data/Mandarin_talkers",
} 
results_dir = "/data/waris/code/TVTSyn/ICLR26Results/{}".format(model_name)
output_dir = "/data/waris/code/TVTSyn/ICLR26Results_quantitative/{}".format(model_name)
os.makedirs(output_dir, exist_ok=True)

metadata_path = "/data/waris/code/TVTSyn/evaluation/evaluation_data.csv"
metadata_df = pd.read_csv(metadata_path)

In [5]:
selected_metadata_df = metadata_df[metadata_df['target_dataset'] == 'emime-eng']
print("Total # utterances to convert: {}".format(len(selected_metadata_df)))

Total # utterances to convert: 6850


In [6]:
def compute_cosine_similarity(audio_file1, audio_file2, sr=16000):
    audio1, _ = librosa.load(audio_file1, sr=sr)
    audio2, _ = librosa.load(audio_file2, sr=sr)
    
    audio1_em  = torch.from_numpy(audio1).unsqueeze(0).to("cuda" if torch.cuda.is_available() else "cpu")
    audio2_em  = torch.from_numpy(audio2).unsqueeze(0).to("cuda" if torch.cuda.is_available() else "cpu")

    embed1 = speaker_encoder(audio1_em)
    embed2 = speaker_encoder(audio2_em)

    # the resulting embeddings can be used for cosine similarity-based retrieval
    cosine_sim = torch.nn.CosineSimilarity(dim=-1)
    similarity = cosine_sim(embed1, embed2)
    return similarity.item()

In [7]:
result_path = os.path.join(output_dir, 'xv_ecapa_cosine_sim.csv')

if not os.path.exists(result_path):
    result_df = []
    for idx, row in tqdm.tqdm(selected_metadata_df.iterrows(), total=len(selected_metadata_df)):
        source_utterance = os.path.join(data_dir[row['source_dataset']], row['source_utterance'])
        assert os.path.exists(source_utterance), f"Source utterance not found: {source_utterance}"
        target_utterance = os.path.join(data_dir[row['target_dataset']], row['target_utterance'])
        assert os.path.exists(target_utterance), f"Target utterance not found: {target_utterance}"

        vc_utterance = os.path.join(results_dir, os.path.basename(row['target_utterance']).replace('.wav', ''), '{}_{}'.format(row['source_speaker'], os.path.basename(row['source_utterance'])))
        assert os.path.exists(vc_utterance), f"VC utterance not found: {vc_utterance}"

        src_cos_sim = compute_cosine_similarity(source_utterance, vc_utterance)
        tgt_cos_sim = compute_cosine_similarity(target_utterance, vc_utterance)

        result_df.append({
            'source_dataset': row['source_dataset'],
            'source_speaker': row['source_speaker'],
            'source_utterance': source_utterance,
            'target_dataset': row['target_dataset'],
            'target_utterance': target_utterance,
            'src_cos_sim': src_cos_sim,
            'tgt_cos_sim': tgt_cos_sim,
        })

    result_df = pd.DataFrame(result_df)
    result_df.to_csv(result_path, index=False)
else:
    result_df = pd.read_csv(result_path)

100%|██████████| 6850/6850 [12:54<00:00,  8.85it/s]


In [8]:
print("Average source cosine similarity: {:.3f} +/- {:.2f}".format(result_df['src_cos_sim'].mean(), result_df['src_cos_sim'].std()))
print("Average target cosine similarity: {:.3f} +/- {:.2f}".format(result_df['tgt_cos_sim'].mean(), result_df['tgt_cos_sim'].std()))

Average source cosine similarity: 0.472 +/- 0.08
Average target cosine similarity: 0.791 +/- 0.06
